In [ ]:
import requests
import copy
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from sqlalchemy import text
import os
import sys

projectRoot = os.path.abspath(os.path.join(os.getcwd(), '..'))
if projectRoot not in sys.path:
    sys.path.insert(0, projectRoot)
from helpers import initDB

<div class='alert alert-block alert-info'>
<b>Team Games:</b> using a team's link, all their games for the year will be located
</div>

In [ ]:
#### direct graphql request ####
from graphqlQueries import teamFixtureQuery

def directTeamRequest(teamURL, session=None):
    uniqueSession = session is None
    if uniqueSession:
        session = requests.Session()
    graphqlURL = 'https://api.playhq.com/graphql'
    teamID = teamURL.rstrip('/').split('/')[-1]
    headers = {
        "content-type": "application/json",
        "origin": "https://www.playhq.com",
        "referer": teamURL,
        "tenant": "afl",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    }
    payload = {
        "operationName": "teamFixture",
        "variables": {"teamID": teamID},
        "query": teamFixtureQuery
    }
    try:
        response = session.post(graphqlURL, json=payload, headers=headers)
        response.raise_for_status()
        return response.json()['data']
    finally:
        if uniqueSession:
            session.close()
    
#directTeamRequest('https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-c3/bb882c71')

In [ ]:
def parseTeamData(data):
    grade = data['discoverTeam']['grade']['name'].lower().replace(' ', '-')
    rounds = data['discoverTeamFixture']
    gameIDs = []
    for round in rounds:
        if round['fixture']['byes'] == []:
            roundGameID = round['fixture']['games'][0]['id']
            gameIDs.append(roundGameID)
    return {'grade': grade, 'games': gameIDs}

#parseTeamData(data)

<div class='alert alert-block alert-info'>
<b>Game Players</b>
</div>

In [ ]:
from graphqlQueries import gamePlayersQuery

def directGameRequest(url, session=None):
    uniqueSession = session is None
    if uniqueSession:
        session = requests.Session()
    graphqlURL = 'https://api.playhq.com/graphql'
    gameID = url.rstrip('/').split('/')[-1]
    headers = {
        "content-type": "application/json",
        "origin": "https://www.playhq.com",
        "referer": url,
        "tenant": "afl",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    }
    payload = {
        "operationName": "gameView",
        "variables": {
            "gameId": gameID,
            "gameStatisticsFilter": {
                "classification": "TOTAL"
            }
        },
        "query": gamePlayersQuery
    }
    try:
        response = session.post(graphqlURL, json=payload, headers=headers)
        response.raise_for_status()
        return response.json()['data']
    finally:
        if uniqueSession:
            session.close()

#directGameRequest('https://www.playhq.com/afl/org/perth-football-league/perth-football-league-2025/budget-car-and-truck-rental-c3-men/game-centre/fab3fdd5')

In [ ]:
def parseGameData(data):
    data = data['discoverGame']
    side = 'home'
    if data['home']['organisation']['name'] != 'University (Perth Football League)':
        side = 'away'
    players = data['statistics'][side]['players']
    scrapedPlayers = []
    for player in players:
        player = player['player']
        if 'name' in player:
            pass #private player
        else:
            player = player['profile']
            scrapedPlayers.append([player['id'], player['firstName'], player['lastName']])
    return scrapedPlayers

#parseGameData(gameData)

<div class='alert alert-block alert-info'>
<b>Player Career</b>
</div>

In [ ]:
from graphqlQueries import playerHistoryQuery

def directPlayerRequest(url, session=None):
    uniqueSession = session is None
    if uniqueSession:
        session = requests.Session()
    graphqlURL = 'https://api.playhq.com/graphql'
    profileID = url.rstrip('/').split('/')[-2]
    headers = {
        "content-type": "application/json",
        "origin": "https://www.playhq.com",
        "referer": "https://www.playhq.com/",
        "tenant": "afl",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    }
    payload = {
        "operationName": "publicProfileStatistics",
        "variables": {
            "profileID": profileID
        },
        "query": playerHistoryQuery
    }
    try:
        response = session.post(graphqlURL, json=payload, headers=headers)
        response.raise_for_status()
        return response.json()['data']
    finally:
        if uniqueSession:
            session.close()

#directPlayerRequest('https://www.playhq.com/public/profile/490d2f42-daea-43ae-8bf1-b10b0e2cab2b/statistics?tenant=afl')

In [ ]:
def parseProfileData(data):
    clubs = data['publicProfileStatistics']['careerStatistics']['clubStatistics']
    for club in clubs:
        if club['club']['name'] == 'University (Perth Football League)':
            playerStats = club['statistics']
            for stat in playerStats:
                if stat['details']['value'] == 'APPEARANCE':  
                    return int(stat['count'])
    profile = data['publicProfile']
    print(f"Could not find any data for {profile['firstName']} {profile['lastName']} with ID: {profile['id']}")
    return 0

#parseProfileData(profileData)

<div class='alert alert-block alert-success'>
<b>Complete Club Scrape</b>
</div>

In [ ]:
teamLinks2024 = [
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-a/341833cd',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-ar/ee8e0ccb',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-bjc/67e45bef',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-c4/9e89ed49',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-c4r/96ecd53e',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-psc/e064c5d3',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-arw/327fa075',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-wa/ec5f13f7'
]

In [ ]:
teamLinks2025 = [
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-a/a7101275',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-ar/f63bf5dd',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-bjc/ae04ae10',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-c3/bb882c71',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-c3r/89ad8e8e',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-psc/807f5521',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-arw/ec91f6cc',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-wa/d5496ec8',
]

: 

In [ ]:
def clubUniquePlayers(teamLinks):
    uniquePlayers = {}
    with requests.Session() as session:
        for teamLink in teamLinks:
            print(f'Requesting {teamLink}...')
            #locate all games a team partakes in
            rawTeamInfo = directTeamRequest(teamLink, session=session)
            teamInfo = parseTeamData(rawTeamInfo)
            teamGrade = teamInfo['grade']
            teamGames = teamInfo['games']
            for idx, game in enumerate(teamGames):
                print(f'{idx+1}/{len(teamGames)}')
                #locate all uni players in each game
                gameLink = f'https://www.playhq.com/afl/org/perth-football-league/perth-football-league-2025/{teamGrade}/game-centre/{game}'
                rawGameInfo = directGameRequest(gameLink, session=session)
                gameInfo = parseGameData(rawGameInfo)
                for playerID, firstName, lastName in gameInfo:
                    #storing each player within the global dict
                    uniquePlayers[playerID] = [playerID, firstName, lastName]
    return list(uniquePlayers.values())
            
    
#clubPlayers = clubUniquePlayers(teamLinks) 
#requests 3m27.3s
#session+!deepCopy 1m44.7s

def parallelClubUniquePlayers(teamLinks, maxWorkers=8):
    uniquePlayers = {}
    gameLinks = []
    #locating all games 
    with requests.Session() as session:
        for teamIdx, teamLink in enumerate(teamLinks, start=1):
            print(f'Team {teamIdx}/{len(teamLinks)}')
            rawTeamInfo = directTeamRequest(teamLink, session=session)
            teamInfo = parseTeamData(rawTeamInfo)
            teamGrade = teamInfo['grade']
            teamGames = teamInfo['games']
            for game in teamGames:
                gameLink = f'https://www.playhq.com/afl/org/perth-football-league/perth-football-league-2025/{teamGrade}/game-centre/{game}'
                gameLinks.append(gameLink)
    print(f'Queued {len(gameLinks)} games...')
    #finding all university players who participated in matches
    with ThreadPoolExecutor(max_workers=maxWorkers) as executor:
        futures = {
            executor.submit(directGameRequest, gameLink): gameLink for gameLink in gameLinks #mustn't share session across threads
        }
        for idx, future in enumerate(as_completed(futures), start=1):
            gameLink = futures[future]
            print(f'Game {idx}/{len(gameLinks)}')
            rawGameInfo = future.result()
            gameInfo = parseGameData(rawGameInfo)
            for playerID, firstName, lastName in gameInfo:
                uniquePlayers[playerID] = [playerID, firstName, lastName]
    return list(uniquePlayers.values())

clubPlayers = parallelClubUniquePlayers(teamLinks2025, maxWorkers=8) 
#38.8s/8wkrs 

In [ ]:
def playerCounts(clubPlayers):
    players = copy.deepcopy(clubPlayers) #for testing, remove in prod
    uniquePlayers = {}
    for idx, player in enumerate(players):
        print(f'{idx+1}/{len(players)}')
        playerID, firstName, lastName = player
        rawData = directPlayerRequest(f'https://www.playhq.com/public/profile/{playerID}/statistics?tenant=afl')
        count = parseProfileData(rawData)
        uniquePlayers[playerID] = [playerID, firstName, lastName, count]
    return list(uniquePlayers.values())

#completePlayers = playerCounts(clubPlayers) #4m57s

def parallelPlayerCounts(clubPlayers, maxWorkers=8):
    uniquePlayers = {}
    with ThreadPoolExecutor(max_workers=maxWorkers) as executor:
        futures = {
            executor.submit(
                directPlayerRequest, f'https://www.playhq.com/public/profile/{playerID}/statistics?tenant=afl'
            ) : (playerID, firstName, lastName) 
            for playerID, firstName, lastName in clubPlayers
        }
        for idx, future in enumerate(as_completed(futures), start=1):
            playerID, firstName, lastName = futures[future]
            print(f'{idx}/{len(clubPlayers)}')
            rawData = future.result()
            count = parseProfileData(rawData)
            uniquePlayers[playerID] = [playerID, firstName, lastName, count]
    return list(uniquePlayers.values())

parallelPlayers = parallelPlayerCounts(clubPlayers, maxWorkers=8) #1m43.2s

In [ ]:
def uploadDF(completePlayers):
    engine = initDB()
    players = copy.deepcopy(completePlayers) #for testing, remove in prod
    df = pd.DataFrame(players, columns=['ID', 'FirstName', 'LastName', 'GameCount'])
    df = df.sort_values(by='GameCount', ascending=False)
    df.to_sql('players', con=engine, if_exists='replace', index=False)
    return df

uploadDF(parallelPlayers)

In [ ]:
#scratch match doc link
#https://docs.google.com/spreadsheets/u/0/d/1MqsnoAFh-aaAEjxa0CQsL7-73fXRoz31/htmlview?pli=1